In [ ]:
import os
import gc
import pickle
import numpy as np
import pandas as pd
import scipy.special as sc
from scipy.special import logit, logsumexp
from numpy.random import default_rng
from sklearn.metrics import roc_auc_score
from jax import random
import jax.numpy as jnp

from balm import run_balm_hurdle_nuts_from_ZY, BALMHyperParams

In [ ]:
def transform_to_ZY(A, eps=1e-6, z_thresh=0.0):
    Z = np.where(A > z_thresh, 1.0, 0.0)
    A_clipped = np.clip(A, eps, 1.0 - eps)
    Y = logit(A_clipped) * Z
    return Z, Y

def prepare_hcp_for_hurdle(A_raw, make_symmetric=True, zero_diagonal=True, clip_negative=True, scale_mode="global_quantile", scale_q=0.995, z_thresh=0.0,  make_more_zeros=True, zero_out_below=0.02):
    A = np.asarray(A_raw, dtype=np.float64).copy()
    L, n, _ = A.shape

    if make_symmetric:
        A = 0.5 * (A + np.transpose(A, (0, 2, 1)))

    if zero_diagonal:
        for i in range(L):
            np.fill_diagonal(A[i], 0.0)

    if clip_negative:
        A = np.maximum(A, 0.0)

    if scale_mode == "global_quantile":
        vals = A.reshape(-1)
        vals = vals[np.isfinite(vals)]
        s = float(np.quantile(vals, scale_q))
        if not np.isfinite(s) or s <= 0:
            raise ValueError("Invalid scale from quantile.")
    elif scale_mode == "global_max":
        s = float(np.nanmax(A))
        if not np.isfinite(s) or s <= 0:
            raise ValueError("Invalid scale from max.")
    else:
        raise ValueError("Invalid scale_mode.")

    A = A / s
    A = np.clip(A, 0.0, 1.0)

    if make_more_zeros and zero_out_below is not None and zero_out_below > 0:
        A = np.where(A <= float(zero_out_below), 0.0, A)

    Z = np.where(A > float(z_thresh), 1.0, 0.0)
    nz = float(np.asarray(Z).mean())
    return A.astype(np.float32), {"scale": s, "nz_rate": nz}

def vec_upper_np(A):
    A = np.asarray(A)
    n = A.shape[-1]
    iu = np.triu_indices(n, k=1)
    return A[..., iu[0], iu[1]]

def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def build_Q_from_samples(U_samp, gamma_samp, tau_samp=None):
    U_samp = np.asarray(U_samp)
    gamma_samp = np.asarray(gamma_samp)
    S, M, n, K = U_samp.shape
    if tau_samp is None:
        tau_samp = np.ones((S,), dtype=np.float64)
    else:
        tau_samp = np.asarray(tau_samp).astype(np.float64).reshape(-1)

    Q = np.zeros((S, M, n, n), dtype=np.float64)
    for s in range(S):
        tau = float(tau_samp[s])
        for m in range(M):
            U = U_samp[s, m]
            g = gamma_samp[s, m]
            Qsm = tau * ((U * g) @ U.T)
            Qsm = 0.5 * (Qsm + Qsm.T)
            np.fill_diagonal(Qsm, 0.0)
            Q[s, m] = Qsm
    return Q

def posterior_mean_Q_W(samples):
    U_samp = np.array(samples["U"])
    gamma_samp = np.array(samples["gamma"])
    tau_samp = np.array(samples.get("tau", None))
    Q_samp = build_Q_from_samples(U_samp, gamma_samp, tau_samp=tau_samp)
    Q_hat = Q_samp.mean(axis=0)
    W_hat = np.array(samples["W"]).mean(axis=0)
    return Q_hat, W_hat

def student_t_logpdf(y, df, mu, sigma):
    const = sc.gammaln((df + 1.0) / 2.0) - sc.gammaln(df / 2.0) - 0.5 * np.log(df * np.pi) - np.log(sigma)
    return const - ((df + 1.0) / 2.0) * np.log1p(((y - mu) / sigma)**2 / df)

def normal_logpdf(y, mu, sigma):
    return -0.5 * np.log(2 * np.pi * sigma * sigma) - 0.5 * ((y - mu) ** 2) / (sigma * sigma)

def bernoulli_logpmf(z, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return z * np.log(p) + (1 - z) * np.log1p(-p)

def waic_hurdle_fast(samples, A, eps=1e-6, z_thresh=0.0, chunk_edges=200, max_draws=150, is_student_t=False, t_df=5.0):
    Z_obs, Y_obs = transform_to_ZY(A.astype(np.float32), eps=eps, z_thresh=z_thresh)
    Z_obs = np.asarray(Z_obs, dtype=np.float64)
    Y_obs = np.asarray(Y_obs, dtype=np.float64)

    W = np.asarray(samples["W"], dtype=np.float64)
    a0 = np.asarray(samples["a0"], dtype=np.float64)
    a1 = np.asarray(samples["a1"], dtype=np.float64)
    sig = np.sqrt(np.asarray(samples["sigma2"], dtype=np.float64))

    S = W.shape[0]
    if max_draws is not None and S > max_draws:
        idx = np.linspace(0, S-1, int(max_draws)).astype(int)
        W, a0, a1, sig = W[idx], a0[idx], a1[idx], sig[idx]
        S = W.shape[0]
    else:
        idx = slice(None)

    U = np.asarray(samples["U"], dtype=np.float64)[idx]
    gamma = np.asarray(samples["gamma"], dtype=np.float64)[idx]
    tau = np.asarray(samples.get("tau", np.ones((U.shape[0],), dtype=np.float64)), dtype=np.float64)[idx]

    Q_draws = build_Q_from_samples(U, gamma, tau_samp=tau)
    Qut = vec_upper_np(Q_draws)

    L, P = Z_obs.shape
    lppd_sum = 0.0
    pwaic_sum = 0.0

    for p0 in range(0, P, int(chunk_edges)):
        p1 = min(P, p0 + int(chunk_edges))
        cols = slice(p0, p1)
        Zc = Z_obs[:, cols]
        Yc = Y_obs[:, cols]
        ll_chunk = np.zeros((S, L * (p1 - p0)), dtype=np.float64)

        for s in range(S):
            mu = W[s] @ Qut[s, :, cols]
            pi = sigmoid_np(a0[s] + a1[s] * mu)
            llz = bernoulli_logpmf(Zc, pi)
            if is_student_t:
                lly = student_t_logpdf(Yc, t_df, mu, float(sig[s])) * Zc
            else:
                lly = normal_logpdf(Yc, mu, float(sig[s])) * Zc
            ll_chunk[s] = (llz + lly).reshape(-1)

        lppd_point = logsumexp(ll_chunk, axis=0) - np.log(S)
        var_point = np.var(ll_chunk, axis=0, ddof=1)
        lppd_sum += float(np.sum(lppd_point))
        pwaic_sum += float(np.sum(var_point))

    waic = float(-2.0 * (lppd_sum - pwaic_sum))
    return {"WAIC": waic, "lppd": float(lppd_sum), "p_waic": float(pwaic_sum), "draws_used": int(S)}

def auc_Z_subsample(samples, A, eps=1e-6, z_thresh=0.0, N=20000, seed=123, max_draws=200):
    rng = default_rng(seed)
    Z_obs, _ = transform_to_ZY(A.astype(np.float32), eps=eps, z_thresh=z_thresh)
    Z_obs = np.asarray(Z_obs, dtype=np.int32)
    L, P = Z_obs.shape
    total = L * P
    N = int(min(N, total))

    flat = rng.choice(total, size=N, replace=False)
    ell = flat // P
    pidx = flat % P
    y = Z_obs[ell, pidx]
    if y.min() == y.max():
        return np.nan

    W = np.asarray(samples["W"], dtype=np.float64)
    a0 = np.asarray(samples["a0"], dtype=np.float64)
    a1 = np.asarray(samples["a1"], dtype=np.float64)
    S = W.shape[0]

    if max_draws is not None and S > max_draws:
        idx = np.linspace(0, S-1, int(max_draws)).astype(int)
        W, a0, a1 = W[idx], a0[idx], a1[idx]
        S = W.shape[0]
    else:
        idx = slice(None)

    U = np.asarray(samples["U"], dtype=np.float64)[idx]
    gamma = np.asarray(samples["gamma"], dtype=np.float64)[idx]
    tau = np.asarray(samples.get("tau", np.ones((U.shape[0],), dtype=np.float64)), dtype=np.float64)[idx]

    Q_draws = build_Q_from_samples(U, gamma, tau_samp=tau)
    Qut = vec_upper_np(Q_draws)

    M_dim = W.shape[2]
    mu_s = np.zeros((S, N), dtype=np.float64)
    for m in range(M_dim):
        mu_s += W[:, ell, m] * Qut[:, m, pidx]

    pi = sigmoid_np(a0[:, None] + a1[:, None] * mu_s)
    score = pi.mean(axis=0)
    return float(roc_auc_score(y, score))

def summarize_samples_quick(samples):
    out = {}
    for k in ["a0", "a1", "sigma2", "tau"]:
        if k in samples:
            x = np.asarray(samples[k], dtype=float).reshape(-1)
            out[f"{k}_mean"] = float(np.mean(x))

    if "W" in samples:
        W = np.asarray(samples["W"], dtype=float)
        Wm = W.mean(axis=0)
        eps_val = 1e-12
        ent = -np.sum(Wm * np.log(Wm + eps_val), axis=1)
        out["W_entropy_mean"] = float(ent.mean())
        out["W_max_mean"] = float(np.max(Wm, axis=1).mean())
    return out

In [ ]:
data_path = "HCP_connectomes_all_68.npz"
data = np.load(data_path)
keys = list(data.files)
L_subj = len(keys)
n_nodes = 68

A_raw = np.zeros((L_subj, n_nodes, n_nodes), dtype=np.float64)
for i, k in enumerate(keys):
    A_raw[i] = np.asarray(data[k], dtype=np.float64)

eps_val = 1e-6
z_thresh_val = 0.0
zero_out_below_val = 0.02

A_fit, prep_info = prepare_hcp_for_hurdle(
    A_raw,
    scale_mode="global_quantile",
    scale_q=0.995,
    z_thresh=z_thresh_val,
    make_more_zeros=True,
    zero_out_below=zero_out_below_val,
)

with open("hcp_balm_prep_info.pkl", "wb") as f:
    pickle.dump({"prep_info": prep_info, "z_thresh": z_thresh_val, "zero_out_below": zero_out_below_val}, f)

Z_fit, Y_fit = transform_to_ZY(A_fit, eps=eps_val, z_thresh=z_thresh_val)
Z_fit_jax = jnp.array(Z_fit)
Y_fit_jax = jnp.array(Y_fit)

K_val = 4
M_list = [2, 3, 4, 5, 6, 7, 8]

# Fit Gaussian Models

save_prefix_g = "hcp_balm_gauss"
hyper_g = BALMHyperParams(a1_fixed=None, use_student_t=False)

for M_fit in M_list:
    print(f"\nFitting BALM (Gaussian) on HCP with M = {M_fit}")
    key = random.PRNGKey(9000 + int(M_fit))

    samples = run_balm_hurdle_nuts_from_ZY(
        rng_key=key, Z=Z_fit_jax, Y=Y_fit_jax, n=int(n_nodes), M=int(M_fit), K=int(K_val),
        hyper=hyper_g, num_warmup=1000, num_samples=1000, num_chains=1,
    )

    out_pkl = f"{save_prefix_g}_samples_M{int(M_fit)}.pkl"
    with open(out_pkl, "wb") as f:
        pickle.dump(samples, f)

    del samples
    gc.collect()

print("\nCalculating metrics for Gaussian Models...")
rows_g = []
for M_fit in M_list:
    pkl_path = f"{save_prefix_g}_samples_M{int(M_fit)}.pkl"
    with open(pkl_path, "rb") as f:
        samples = pickle.load(f)

    met = summarize_samples_quick(samples)
    met["M_fit"] = int(M_fit)
    met["AUC_Z"] = auc_Z_subsample(samples, A_fit, eps=eps_val, z_thresh=z_thresh_val)

    w = waic_hurdle_fast(samples, A_fit, eps=eps_val, z_thresh=z_thresh_val, is_student_t=False)
    met.update({"WAIC": w["WAIC"], "lppd": w["lppd"], "p_waic": w["p_waic"]})

    rows_g.append(met)
    del samples
    gc.collect()

df_metrics_g = pd.DataFrame(rows_g).sort_values("M_fit")
print("\nFinal Metrics Summary (Gaussian):")
print(df_metrics_g[["M_fit", "WAIC", "AUC_Z", "p_waic", "a0_mean", "a1_mean", "W_max_mean", "W_entropy_mean"]])

In [ ]:
# Fit Student-t Models
save_prefix_t = "hcp_balm_t"
t_df_val = 5.0
hyper_t = BALMHyperParams(a1_fixed=None, use_student_t=True, t_df=t_df_val)

for M_fit in M_list:
    print(f"\nFitting BALM (Student-t) on HCP with M = {M_fit}")
    key = random.PRNGKey(8000 + int(M_fit))

    samples_t = run_balm_hurdle_nuts_from_ZY(
        rng_key=key, Z=Z_fit_jax, Y=Y_fit_jax, n=int(n_nodes), M=int(M_fit), K=int(K_val),
        hyper=hyper_t, num_warmup=500, num_samples=1000, num_chains=1,
        target_accept_prob=0.80, max_tree_depth=8
    )

    out_pkl_t = f"{save_prefix_t}_samples_M{int(M_fit)}.pkl"
    with open(out_pkl_t, "wb") as f:
        pickle.dump(samples_t, f)

    del samples_t
    gc.collect()

print("\nCalculating metrics for Student-t Models...")
rows_t = []
for M_fit in M_list:
    pkl_path = f"{save_prefix_t}_samples_M{int(M_fit)}.pkl"
    with open(pkl_path, "rb") as f:
        samples = pickle.load(f)

    met = summarize_samples_quick(samples)
    met["M_fit"] = int(M_fit)
    met["AUC_Z"] = auc_Z_subsample(samples, A_fit, eps=eps_val, z_thresh=z_thresh_val)

    w = waic_hurdle_fast(samples, A_fit, eps=eps_val, z_thresh=z_thresh_val, is_student_t=True, t_df=t_df_val)
    met.update({"WAIC": w["WAIC"], "lppd": w["lppd"], "p_waic": w["p_waic"]})

    rows_t.append(met)
    del samples
    gc.collect()

df_metrics_t = pd.DataFrame(rows_t).sort_values("M_fit")
print("\nFinal Metrics Summary (Student-t):")
print(df_metrics_t[["M_fit", "WAIC", "AUC_Z", "p_waic", "a0_mean", "a1_mean", "W_max_mean", "W_entropy_mean"]])